In [ ]:
# 1. Install Kaggle library
!pip install -q kaggle

# 2. Download the dataset directly using kagglehub (No API key needed!)
import kagglehub

print("⏬ Downloading PlantVillage dataset...")
path = kagglehub.dataset_download("emmarex/plantdisease")

print("✅ Download complete! Path to data:", path)

⏬ Downloading PlantVillage dataset...
Using Colab cache for faster access to the 'plantdisease' dataset.
✅ Download complete! Path to data: /kaggle/input/plantdisease


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny
from torchvision.transforms import v2
from tqdm.notebook import tqdm

# Setup GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using GPU device: {torch.cuda.get_device_name(0)}")

# --- 1. DATA TRANSFORMS ---
train_transforms = v2.Compose([
    v2.Resize((256, 256)),
    v2.TrivialAugmentWide(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = v2.Compose([
    v2.Resize((256, 256)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- 2. DATASET PREPARATION ---
# Change this path to where your dataset lives in Colab/Drive
DATASET_PATH = os.path.join(path, "PlantVillage")  # or just use `path` if classes are at the root level

ds_train = ImageFolder(root=DATASET_PATH, transform=train_transforms)
ds_val = ImageFolder(root=DATASET_PATH, transform=val_transforms)

num_samples = len(ds_train)
indices = list(range(num_samples))
split = int(np.floor(0.2 * num_samples))

np.random.seed(42)
np.random.shuffle(indices)

train_idx, val_idx = indices[split:], indices[:split]

pv_train_subset = Subset(ds_train, train_idx)
pv_val_subset = Subset(ds_val, val_idx)

# High num_workers and batch size work seamlessly on Nvidia T4
train_loader = DataLoader(
    pv_train_subset, batch_size=64, shuffle=True, drop_last=True, num_workers=2
)
val_loader = DataLoader(
    pv_val_subset, batch_size=64, shuffle=False, num_workers=2
)


# --- 3. CONVNEXT MODEL ---
class PlantHealthConvNeXt(nn.Module):

  def __init__(self, num_classes=38):
    super().__init__()
    weights = ConvNeXt_Tiny_Weights.DEFAULT
    self.backbone = convnext_tiny(weights=weights)

    in_features = self.backbone.classifier[2].in_features
    self.backbone.classifier[2] = nn.Sequential(
        nn.Dropout(p=0.4), nn.Linear(in_features, num_classes)
    )

  def forward(self, x):
    return self.backbone(x)


model = PlantHealthConvNeXt(num_classes=38).to(device)

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(
    [
        {"params": model.backbone.features.parameters(), "lr": 1e-5},
        {"params": model.backbone.classifier.parameters(), "lr": 3e-4},
    ],
    weight_decay=1e-2,
)

EPOCHS = 5
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

# --- 4. FAST CUDA TRAINING LOOP ---
best_acc = 0.0

for epoch in range(EPOCHS):
  model.train()
  train_loss, train_acc = 0.0, 0.0

  for X, y in tqdm(
      train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False
  ):
    X, y = X.to(device), y.to(device)

    optimizer.zero_grad()
    y_pred = model(X)
    loss = loss_fn(y_pred, y)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    train_loss += loss.item()
    train_acc += (y_pred.argmax(dim=1) == y).sum().item() / len(y)

  train_loss /= len(train_loader)
  train_acc /= len(train_loader)

  # Validation Phase
  model.eval()
  val_loss, val_acc = 0.0, 0.0
  with torch.inference_mode():
    for X, y in tqdm(
        val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False
    ):
      X, y = X.to(device), y.to(device)
      val_pred = model(X)
      loss = loss_fn(val_pred, y)

      val_loss += loss.item()
      val_acc += (val_pred.argmax(dim=1) == y).sum().item() / len(y)

    val_loss /= len(val_loader)
    val_acc /= len(val_loader)

  scheduler.step()

  print(
      f"Epoch {epoch+1:02d}/{EPOCHS:02d} | Train Acc: {train_acc*100:.2f}% | Val"
      f" Acc: {val_acc*100:.2f}%"
  )

  if val_acc > best_acc:
    best_acc = val_acc
    torch.save(
        model.state_dict(), "plantvillage_convnext_pretrained.pth"
    )
    print(f"🔥 Model Saved! (Val Acc: {best_acc*100:.2f}%)\n")

🚀 Using GPU device: Tesla T4
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 189MB/s] 


Epoch 1/5 [Train]:   0%|          | 0/257 [00:00<?, ?it/s]

Epoch 1/5 [Val]:   0%|          | 0/65 [00:00<?, ?it/s]

Epoch 01/05 | Train Acc: 72.34% | Val Acc: 95.12%
🔥 Model Saved! (Val Acc: 95.12%)



...

Epoch 4/5 [Val]:   0%|          | 0/65 [00:00<?, ?it/s]

Epoch 04/05 | Train Acc: 97.31% | Val Acc: 98.92%
🔥 Model Saved! (Val Acc: 98.92%)



Epoch 5/5 [Train]:   0%|          | 0/257 [00:00<?, ?it/s]

Epoch 5/5 [Val]:   0%|          | 0/65 [00:00<?, ?it/s]

Epoch 05/05 | Train Acc: 97.91% | Val Acc: 99.01%
🔥 Model Saved! (Val Acc: 99.01%)



In [ ]:
DRIVE_MODEL_DIR = '/content/drive/MyDrive/PlantModels'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    DRIVE_MODEL_DIR, 'plantvillage_convnext_pretrained.pth'
)

# 3. Inside your training loop, update the save line:
if val_acc > best_acc:
  best_acc = val_acc
  torch.save(model.state_dict(), SAVE_PATH)
  print(f'🔥 Checkpoint saved directly to Drive: {SAVE_PATH}')

In [ ]:
from google.colab import files

# Download directly from local Colab runtime workspace
files.download("plantvillage_convnext_pretrained.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import os
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny
from torchvision.transforms import v2
from tqdm.notebook import tqdm
from google.colab import drive

# 1. Drive Mount & Device Setup
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 2. Transforms
train_transforms_256 = v2.Compose([
    v2.Resize((256, 256)),
    v2.TrivialAugmentWide(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms_256 = v2.Compose([
    v2.Resize((256, 256)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Custom Dataset
class CustomPlantDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)

        if (self.root_dir / "images").exists():
            self.images_dir = self.root_dir / "images"
            self.labels_dir = self.root_dir / "labels"
        else:
            self.images_dir = self.root_dir
            self.labels_dir = self.root_dir

        self.transform = transform

        if not self.images_dir.exists():
            raise FileNotFoundError(f"Directory not found: {self.images_dir}")

        self.image_files = sorted([
            f for f in os.listdir(self.images_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])
        print(f"✅ Found {len(self.image_files)} images in '{self.images_dir}'")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = self.images_dir / img_name
        image = Image.open(img_path).convert("RGB")

        label_name = img_name.rsplit(".", 1)[0] + ".txt"
        label_path = self.labels_dir / label_name

        label = 0
        if label_path.exists() and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                first_line = f.readline().strip().split()
                if first_line:
                    label = int(first_line[0])

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# 4. Model Architecture Definition FIRST
class PlantHealthConvNeXt(nn.Module):
    def __init__(self, num_classes=38):
        super().__init__()
        weights = ConvNeXt_Tiny_Weights.DEFAULT
        self.backbone = convnext_tiny(weights=weights)

        in_features = self.backbone.classifier[2].in_features
        self.backbone.classifier[2] = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

# 5. Model Instantiation AFTER definition and device setup
model = PlantHealthConvNeXt(num_classes=38).to(device)
print("✅ Model successfully initialized on GPU!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Using Device: cuda (Tesla T4)
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 134MB/s]


✅ Model successfully initialized on GPU!


In [4]:
def train_colab(
    model,
    loss_fn,
    optimizer,
    EPOCHS,
    scheduler,
    save_model_name="best_health_model_v3.pth",  # <-- Added quotes here
):
  DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/Plant project on google"

  PLANTDOC_TRAIN_DIR = os.path.join(
      DRIVE_PROJECT_ROOT, "Plant_project_data/train"
  )
  PLANTDOC_VAL_DIR = os.path.join(
      DRIVE_PROJECT_ROOT, "Plant_project_data/valid"
  )

  # Load warm weights if starting fresh or continuing
  PRETRAINED_PATH_DRIVE = os.path.join(DRIVE_PROJECT_ROOT, save_model_name)
  if os.path.exists(PRETRAINED_PATH_DRIVE):
    model.load_state_dict(
        torch.load(PRETRAINED_PATH_DRIVE, map_location=device)
    )
    print(f"✅ Loaded checkpoint weights from: {PRETRAINED_PATH_DRIVE}")

  train_ds = CustomPlantDataset(
      PLANTDOC_TRAIN_DIR, transform=train_transforms_256
  )
  val_ds = CustomPlantDataset(PLANTDOC_VAL_DIR, transform=val_transforms_256)

  train_loader = DataLoader(
      train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True
  )
  val_loader = DataLoader(
      val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True
  )

  best_val_acc = 0.4523  # Starting baseline from your 12-epoch run

  for epoch in range(EPOCHS):
    model.train()
    train_loss, train_acc = 0.0, 0.0
    for X, y in tqdm(
        train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False
    ):
      X, y = X.to(device), y.to(device)
      optimizer.zero_grad()
      y_pred = model(X)
      loss = loss_fn(y_pred, y)
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      optimizer.step()

      train_loss += loss.item()
      train_acc += (y_pred.argmax(dim=1) == y).sum().item() / len(y)

    train_loss /= len(train_loader)
    train_acc /= len(train_loader)

    model.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.inference_mode():
      for X, y in tqdm(
          val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False
      ):
        X, y = X.to(device), y.to(device)
        val_pred = model(X)
        loss = loss_fn(val_pred, y)
        val_loss += loss.item()
        val_acc += (val_pred.argmax(dim=1) == y).sum().item() / len(y)

      val_loss /= len(val_loader)
      val_acc /= len(val_loader)

    scheduler.step()

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS:02d} | Train Acc: {train_acc*100:.2f}% |"
        f" Val Acc: {val_acc*100:.2f}%"
    )

    if val_acc > best_val_acc:
      best_val_acc = val_acc
      SAVE_PATH = os.path.join(
          DRIVE_PROJECT_ROOT, save_model_name
      )  # <-- Updated to use save_model_name
      torch.save(model.state_dict(), SAVE_PATH)
      print(f"🔥 Best model saved directly to Google Drive: {SAVE_PATH}\n")

In [5]:
EPOCHS = 30

optimizer_1 = torch.optim.AdamW([
    {'params': model.backbone.features.parameters(), 'lr': 1e-5},
    {'params': model.backbone.classifier.parameters(), 'lr': 3e-4}
], weight_decay=1e-2)

scheduler_1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_1, T_max=EPOCHS, eta_min=1e-6)

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Run 30-epoch fine-tuning
train_colab(
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer_1,
    EPOCHS=EPOCHS,
    scheduler=scheduler_1
)

✅ Loaded checkpoint weights from: /content/drive/MyDrive/Plant project on google/best_health_model_v3.pth
✅ Found 1979 images in '/content/drive/MyDrive/Plant project on google/Plant_project_data/train/images'
✅ Found 349 images in '/content/drive/MyDrive/Plant project on google/Plant_project_data/valid/images'


Epoch 1/30 [Train]:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 1/30 [Val]:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 01/30 | Train Acc: 86.27% | Val Acc: 73.27%
🔥 Best model saved directly to Google Drive: /content/drive/MyDrive/Plant project on google/best_health_model_v3.pth



Epoch 2/30 [Train]:   0%|          | 0/31 [00:00<?, ?it/s]

...

Epoch 29/30 [Val]:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 29/30 | Train Acc: 95.00% | Val Acc: 73.27%


Epoch 30/30 [Train]:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 30/30 [Val]:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 30/30 | Train Acc: 95.56% | Val Acc: 73.27%


In [6]:
import os
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny
from torchvision.transforms import v2
from tqdm.notebook import tqdm

# ==========================================
# 1. TRANSFORMS FOR 384x384 RESOLUTION
# ==========================================
train_transforms_384 = v2.Compose([
    v2.Resize((384, 384)),
    v2.TrivialAugmentWide(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms_384 = v2.Compose([
    v2.Resize((384, 384)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [9]:
model_384 = PlantHealthConvNeXt(num_classes=38).to(device)

EPOCHS = 30

# 2. Setup Optimizer & Scheduler
optimizer_384 = torch.optim.AdamW(
    [
        {'params': model_384.backbone.features.parameters(), 'lr': 1e-5},
        {'params': model_384.backbone.classifier.parameters(), 'lr': 3e-4},
    ],
    weight_decay=1e-2,
)

scheduler_384 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_384, T_max=EPOCHS, eta_min=1e-6
)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# 3. Call train_colab with custom model save name
train_colab(
    model=model_384,
    loss_fn=loss_fn,
    optimizer=optimizer_384,
    EPOCHS=EPOCHS,
    scheduler=scheduler_384,
    save_model_name="plantdoc_convnext_384x384_best.pth"
)

✅ Found 1979 images in '/content/drive/MyDrive/Plant project on google/Plant_project_data/train/images'
✅ Found 349 images in '/content/drive/MyDrive/Plant project on google/Plant_project_data/valid/images'


Epoch 1/30 [Train]:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 1/30 [Val]:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 01/30 | Train Acc: 12.29% | Val Acc: 30.48%


Epoch 2/30 [Train]:   0%|          | 0/31 [00:00<?, ?it/s]

...

Epoch 29/30 [Val]:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 29/30 | Train Acc: 84.54% | Val Acc: 72.17%


Epoch 30/30 [Train]:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 30/30 [Val]:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 30/30 | Train Acc: 84.72% | Val Acc: 71.65%
